In [ ]:
import os, shutil, subprocess, sys, time, yaml
REPO = "/kaggle/temp/repo"
os.makedirs("/kaggle/temp", exist_ok=True)
sh = lambda c: subprocess.run(c, shell=True, check=True)
subprocess.run(["git", "clone", "-q", "https://github.com/yanis-ghazi/fred-box-propagation", REPO], check=True)
os.chdir(REPO)
SMOKE = False
if SMOKE:
    p = yaml.safe_load(open("params.yaml"))
    p["select"].update(n_sequences=2, clip_length=40)
    p["simulate"].update(budgets=[1, 2], random_seeds=[0])
    yaml.safe_dump(p, open("params.yaml", "w"), sort_keys=False)
t0 = time.time()
sh(f"SAM2_BUILD_CUDA=0 {sys.executable} -m pip install -q --no-deps --no-build-isolation "
   "git+https://github.com/facebookresearch/sam2.git@2b90b9f5ceec907a1c18123530e92e794ad901a4")
sh(f"{sys.executable} -m pip install -q 'hydra-core>=1.3.2' 'iopath>=0.1.10' dvc -e .")
print("SETUP_S", round(time.time() - t0), flush=True)
for stage in ["index", "select", "fetch", "represent", "propagate", "report"]:
    t0 = time.time()
    sh(f"dvc repro -s {stage}")
    print("STAGE", stage, round(time.time() - t0), flush=True)
    out = "/kaggle/working/out"
    os.makedirs(out, exist_ok=True)
    for d in ["results", "figures"]:
        if os.path.isdir(d):
            shutil.copytree(d, f"{out}/{d}", dirs_exist_ok=True)
    for f in ["dvc.lock", "data/selection.csv", "data/index.csv", "params.yaml"]:
        if os.path.exists(f):
            shutil.copy(f, out)
for f in ["data/selection.csv", "results/representations.csv", "results/timing.csv", "results/precision_check.json",
          "results/quality_by_representation.csv", "results/drift_auroc.csv", "results/simulation_summary.csv",
          "results/metrics.json", "dvc.lock"]:
    if os.path.exists(f):
        print(f"FILE {f}\n" + open(f).read(), flush=True)
